# Step 2 — Prepare the Data (Parte 2)
## Etapa 2.3 — Pré-processamento de Dados

**Projeto:** Tech Challenge Fase 1 — Desafio B — Detecção de Pneumonia  
**Metodologia:** ML Life Cycle — Step 2: Prepare the Data (Pré-processamento)  

---

### Objetivo

Transformar o dataset clínico bruto em conjuntos prontos para treinamento, seguindo as decisões tomadas na EDA (Etapa 2.2):

1. Remover colunas sem valor preditivo (`patient_id`, `timestamp`, `note_sequence`, `clinical_note`, `uncertainty_score`)
2. Aplicar **One-Hot Encoding** na feature categórica `chest_xray_result`
3. Realizar **split por `patient_id`** (treino 70% / validação 15% / teste 15%) para evitar data leakage
4. Aplicar **StandardScaler** com `fit` exclusivamente nos dados de treino
5. Salvar splits em `data/tabular/` e scaler em `models/scaler.pkl`

---

## 1. Importações

In [1]:
import os
import pickle

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print('Bibliotecas carregadas com sucesso.')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

Bibliotecas carregadas com sucesso.
pandas  : 3.0.3
numpy   : 2.4.4


---

## 2. Constantes

In [2]:
RANDOM_STATE = 42
DATA_PATH    = '../data/tabular/'
MODEL_PATH   = '../models/'
TARGET       = 'true_label'

COLUNAS_DESCARTAR  = ['patient_id', 'timestamp', 'note_sequence', 'clinical_note', 'uncertainty_score']
FEATURE_CATEGORICA = 'chest_xray_result'
FEATURES_ESCALAR   = ['oxygen_saturation', 'wbc_count']

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

print('Constantes definidas.')
print(f'  Colunas a descartar : {COLUNAS_DESCARTAR}')
print(f'  Feature categórica  : {FEATURE_CATEGORICA}')
print(f'  Features a escalar  : {FEATURES_ESCALAR}')
print(f'  Split               : {TRAIN_RATIO:.0%} treino / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} teste')

Constantes definidas.
  Colunas a descartar : ['patient_id', 'timestamp', 'note_sequence', 'clinical_note', 'uncertainty_score']
  Feature categórica  : chest_xray_result
  Features a escalar  : ['oxygen_saturation', 'wbc_count']
  Split               : 70% treino / 15% val / 15% teste


---

## 3. Carregamento do Dataset

In [3]:
df = pd.read_csv(f'{DATA_PATH}clinical_pneumonia_dataset.csv')

print(f'Dataset carregado.')
print(f'  Shape     : {df.shape}')
print(f'  Pacientes : {df["patient_id"].nunique():,} únicos')
print(f'  Classes   : {df[TARGET].value_counts().to_dict()}')
df.head(3)

Dataset carregado.
  Shape     : (1500, 12)
  Pacientes : 297 únicos
  Classes   : {'pneumonia': 500, 'pulmonary edema': 500, 'atelectasis': 500}


,patient_id,timestamp,note_sequence,clinical_note,fever,tachycardia,crackles,oxygen_saturation,wbc_count,chest_xray_result,true_label,uncertainty_score
0,PNE-6733,2022-06-11,0,Pyrexia. rapid heart rate. rales. hypoxemia. p...,1,1,1,93.5,7.5,consolidation,pneumonia,0.77
1,PNE-6733,2022-06-12,1,Fever. tachycardia. rales. spo2 < 94%. product...,1,1,1,93.5,7.5,consolidation,pneumonia,0.85
2,PNE-6733,2022-06-13,2,Raised body temp. tachycardia. crackles. spo2 ...,1,1,1,93.5,7.5,consolidation,pneumonia,0.96


---

## 4. Remoção de Colunas sem Valor Preditivo

As seguintes colunas foram identificadas na EDA como **sem valor para o modelo ML clássico**:

| Coluna | Motivo do descarte |
|---|---|
| `patient_id` | Identificador — não é feature clínica |
| `timestamp` | Data da nota clínica — sem valor preditivo direto |
| `note_sequence` | Número da consulta — ordinal sem significado clínico para classificação |
| `clinical_note` | Texto livre — fora do escopo do ML clássico (exigiria NLP) |
| `uncertainty_score` | Meta-feature de geração do dataset sintético — risco de data leakage |

In [4]:
df_clean = df.drop(columns=COLUNAS_DESCARTAR)

print(f'Colunas antes da remoção : {df.shape[1]}')
print(f'Colunas após a remoção   : {df_clean.shape[1]}')
print(f'\nFeatures mantidas:')
for col in df_clean.columns:
    print(f'  {col:<22} → {df_clean[col].dtype}')

Colunas antes da remoção : 12
Colunas após a remoção   : 7

Features mantidas:
  fever                  → int64
  tachycardia            → int64
  crackles               → int64
  oxygen_saturation      → float64
  wbc_count              → float64
  chest_xray_result      → str
  true_label             → str


---

## 5. One-Hot Encoding: `chest_xray_result`

A coluna `chest_xray_result` possui **5 categorias nominais** sem ordem natural: `infiltrate`, `normal`, `consolidation`, `opacity`, `effusion`.

**Estratégia: One-Hot Encoding com `drop_first=True`**

- **Label Encoding seria inadequado** aqui porque implicaria uma ordem artificial entre os achados radiológicos (consolidation=2 > infiltrate=1 > ...), o que não existe clinicamente.
- **One-Hot** cria uma coluna binária por categoria — sem relação de ordem.
- **`drop_first=True`** remove a categoria `consolidation` (primeira em ordem alfabética), eliminando multicolinearidade entre as colunas dummy. O modelo a recupera pelo complemento das demais.

In [5]:
df_encoded = pd.get_dummies(df_clean, columns=[FEATURE_CATEGORICA], drop_first=True, dtype=int)

print(f'Shape antes  : {df_clean.shape}')
print(f'Shape depois : {df_encoded.shape}')

ohe_cols = [c for c in df_encoded.columns if c.startswith(FEATURE_CATEGORICA)]
print(f'\nColunas One-Hot geradas ({len(ohe_cols)}):  ', ohe_cols)
print(f'Categoria de referência removida   : chest_xray_result_consolidation')
df_encoded.head(3)

Shape antes  : (1500, 7)
Shape depois : (1500, 10)

Colunas One-Hot geradas (4):   ['chest_xray_result_effusion', 'chest_xray_result_infiltrate', 'chest_xray_result_normal', 'chest_xray_result_opacity']
Categoria de referência removida   : chest_xray_result_consolidation


,fever,tachycardia,crackles,oxygen_saturation,wbc_count,true_label,chest_xray_result_effusion,chest_xray_result_infiltrate,chest_xray_result_normal,chest_xray_result_opacity
0,1,1,1,93.5,7.5,pneumonia,0,0,0,0
1,1,1,1,93.5,7.5,pneumonia,0,0,0,0
2,1,1,1,93.5,7.5,pneumonia,0,0,0,0


---

## 6. Separação de Features e Target

O target `true_label` é **multi-classe** com 3 categorias: `pneumonia`, `pulmonary edema` e `atelectasis`.

**Encoding do target:** mantido como string. scikit-learn suporta nativamente strings em classificadores multi-classe — não há necessidade de `LabelEncoder` neste ponto.

> O dataset está **perfeitamente balanceado** (500 registros por classe), conforme confirmado na EDA. Não há necessidade de oversampling (SMOTE) nem `class_weight='balanced'`.

In [6]:
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f'X — features : {X.shape}')
print(f'  Colunas: {X.columns.tolist()}')
print(f'\ny — target   : {y.shape}')
print(f'  Classes: {y.unique().tolist()}')
print(f'\nDistribuição do target:')
display(y.value_counts().rename('Registros').to_frame())

X — features : (1500, 9)
  Colunas: ['fever', 'tachycardia', 'crackles', 'oxygen_saturation', 'wbc_count', 'chest_xray_result_effusion', 'chest_xray_result_infiltrate', 'chest_xray_result_normal', 'chest_xray_result_opacity']

y — target   : (1500,)
  Classes: ['pneumonia', 'pulmonary edema', 'atelectasis']

Distribuição do target:


,Registros
true_label,
pneumonia,500
pulmonary edema,500
atelectasis,500


---

## 7. Split Treino / Validação / Teste por `patient_id`

### Por que split por paciente e não por registro?

O dataset possui **297 pacientes únicos**, cada um com múltiplas notas clínicas sequenciais. Um `train_test_split` padrão (por registro) poderia alocar notas do **mesmo paciente** em conjuntos diferentes, gerando **data leakage**: o modelo seria treinado e testado em notas do mesmo indivíduo, inflacionando artificialmente as métricas.

**Solução:** Dividir os **297 pacientes** (não os 1.500 registros) em treino/validação/teste, e então selecionar todos os registros de cada grupo:

| Conjunto | Pacientes | Registros esperados |
|---|---|---|
| Treino | ~208 (70%) | ~1.040 (69%) |
| Validação | ~45 (15%) | ~225 (15%) |
| Teste | ~44 (15%) | ~220 (15%) |

`stratify` é aplicado por paciente para manter a proporção de classes nos três conjuntos.

In [7]:
# Um registro por paciente (para fazer o split nos pacientes, não nos registros)
pacientes_df = df[['patient_id', TARGET]].drop_duplicates(subset='patient_id')
paciente_ids    = pacientes_df['patient_id'].values
paciente_labels = pacientes_df[TARGET].values

print(f'Total de pacientes únicos: {len(paciente_ids)}')
print(f'Distribuição: {pd.Series(paciente_labels).value_counts().to_dict()}')

# Split 1: treino (70%) vs restantes (30%)
pids_train, pids_temp, _, y_temp_strat = train_test_split(
    paciente_ids, paciente_labels,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_STATE,
    stratify=paciente_labels
)

# Split 2: validação (50% dos 30%) e teste (50% dos 30%) → 15%/15% do total
pids_val, pids_test, _, _ = train_test_split(
    pids_temp, y_temp_strat,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp_strat
)

print(f'\nPacientes por conjunto:')
print(f'  Treino    : {len(pids_train):>4} ({len(pids_train)/len(paciente_ids):.1%})')
print(f'  Validação : {len(pids_val):>4} ({len(pids_val)/len(paciente_ids):.1%})')
print(f'  Teste     : {len(pids_test):>4} ({len(pids_test)/len(paciente_ids):.1%})')

Total de pacientes únicos: 297
Distribuição: {'atelectasis': 100, 'pulmonary edema': 99, 'pneumonia': 98}

Pacientes por conjunto:
  Treino    :  207 (69.7%)
  Validação :   45 (15.2%)
  Teste     :   45 (15.2%)


In [8]:
# Re-adicionar patient_id temporariamente para filtrar registros
df_com_pid = df_encoded.copy()
df_com_pid['patient_id'] = df['patient_id']

mask_train = df_com_pid['patient_id'].isin(pids_train)
mask_val   = df_com_pid['patient_id'].isin(pids_val)
mask_test  = df_com_pid['patient_id'].isin(pids_test)

X_train = df_com_pid[mask_train].drop(columns=[TARGET, 'patient_id']).reset_index(drop=True)
X_val   = df_com_pid[mask_val].drop(columns=[TARGET, 'patient_id']).reset_index(drop=True)
X_test  = df_com_pid[mask_test].drop(columns=[TARGET, 'patient_id']).reset_index(drop=True)

y_train = df_com_pid[mask_train][TARGET].reset_index(drop=True)
y_val   = df_com_pid[mask_val][TARGET].reset_index(drop=True)
y_test  = df_com_pid[mask_test][TARGET].reset_index(drop=True)

print('Tamanho dos conjuntos (registros):')
print(f'  Treino    : {X_train.shape}  ({len(X_train)/len(df):.1%} dos registros)')
print(f'  Validação : {X_val.shape}   ({len(X_val)/len(df):.1%} dos registros)')
print(f'  Teste     : {X_test.shape}   ({len(X_test)/len(df):.1%} dos registros)')

print('\nDistribuição do target por conjunto:')
for nome, y_serie in [('Treino', y_train), ('Validação', y_val), ('Teste', y_test)]:
    dist = y_serie.value_counts(normalize=True).round(3)
    linha = '  |  '.join([f'{k}: {v:.1%}' for k, v in dist.items()])
    print(f'  {nome:<10}: {linha}')

Tamanho dos conjuntos (registros):
  Treino    : (1050, 9)  (70.0% dos registros)
  Validação : (225, 9)   (15.0% dos registros)
  Teste     : (225, 9)   (15.0% dos registros)

Distribuição do target por conjunto:
  Treino    : pneumonia: 33.3%  |  pulmonary edema: 33.3%  |  atelectasis: 33.3%
  Validação : pneumonia: 33.3%  |  pulmonary edema: 33.3%  |  atelectasis: 33.3%
  Teste     : pneumonia: 33.3%  |  pulmonary edema: 33.3%  |  atelectasis: 33.3%


**Resultado esperado:** As três classes devem aparecer com proporção ~33,3% em cada conjunto, graças ao `stratify` aplicado no split por paciente. Isso confirma que o balanceamento do dataset foi preservado nos três subconjuntos.

---

## 8. Normalização: StandardScaler

Apenas as features numéricas contínuas precisam de normalização:

| Feature | Tipo | Normalizar? | Motivo |
|---|---|---|---|
| `fever`, `tachycardia`, `crackles` | Binárias (0/1) | Não | Já em escala unitária |
| `oxygen_saturation`, `wbc_count` | Contínuas | **Sim** | Escalas diferentes (90-98% vs 6-11 K/µL) |
| Colunas One-Hot (`chest_xray_*`) | Binárias (0/1) | Não | Já em escala unitária |

**Regra crítica — `fit` apenas no treino:**

O `StandardScaler` aprende a média e o desvio padrão durante o `fit()`. Aplicar `fit` no dataset inteiro (incluindo validação e teste) vaza informações dos dados de avaliação para o scaler — **data leakage**. A ordem correta é:

```
X_train_scaled = scaler.fit_transform(X_train)   ← aprende + aplica
X_val_scaled   = scaler.transform(X_val)          ← só aplica
X_test_scaled  = scaler.transform(X_test)         ← só aplica
```

> **Por que normalizar apenas `oxygen_saturation` e `wbc_count`?** Algoritmos baseados em distância (KNN) e modelos lineares (Regressão Logística) são sensíveis à escala. Random Forest e Árvores de Decisão são invariantes à escala, mas o scaler não os prejudica — e manter os dados normalizados simplifica o pipeline de serving na API.

In [9]:
scaler = StandardScaler()

# fit_transform no treino — aprende média e desvio padrão
X_train[FEATURES_ESCALAR] = scaler.fit_transform(X_train[FEATURES_ESCALAR])

# transform em val e teste — aplica a mesma transformação
X_val[FEATURES_ESCALAR]   = scaler.transform(X_val[FEATURES_ESCALAR])
X_test[FEATURES_ESCALAR]  = scaler.transform(X_test[FEATURES_ESCALAR])

print('StandardScaler aplicado com sucesso.')
print(f'\nParâmetros aprendidos no treino (fit apenas nos dados de treino):')
for feat, mean, std in zip(FEATURES_ESCALAR, scaler.mean_, scaler.scale_):
    print(f'  {feat:<22} → média={mean:.4f}  desvio={std:.4f}')

print(f'\nVerificação pós-normalização em X_train (média≈0, desvio≈1):')
display(X_train[FEATURES_ESCALAR].describe().loc[['mean', 'std']].round(4))

StandardScaler aplicado com sucesso.

Parâmetros aprendidos no treino (fit apenas nos dados de treino):
  oxygen_saturation      → média=94.5796  desvio=1.9150
  wbc_count              → média=8.4540  desvio=1.2965

Verificação pós-normalização em X_train (média≈0, desvio≈1):


,oxygen_saturation,wbc_count
mean,0.0000,-0.0000
std,1.0005,1.0005


---

## 9. Salvamento dos Artefatos

In [10]:
# Salvar splits
X_train.to_csv(f'{DATA_PATH}X_train.csv', index=False)
X_val.to_csv(f'{DATA_PATH}X_val.csv', index=False)
X_test.to_csv(f'{DATA_PATH}X_test.csv', index=False)

y_train.to_csv(f'{DATA_PATH}y_train.csv', index=False)
y_val.to_csv(f'{DATA_PATH}y_val.csv', index=False)
y_test.to_csv(f'{DATA_PATH}y_test.csv', index=False)

print('Splits salvos em data/tabular/:')
for nome in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    path = f'{DATA_PATH}{nome}.csv'
    shape = pd.read_csv(path).shape
    print(f'  {nome}.csv → {shape}')

Splits salvos em data/tabular/:
  X_train.csv → (1050, 9)
  X_val.csv → (225, 9)
  X_test.csv → (225, 9)
  y_train.csv → (1050, 1)
  y_val.csv → (225, 1)
  y_test.csv → (225, 1)


In [11]:
# Salvar scaler
os.makedirs(MODEL_PATH, exist_ok=True)
scaler_path = f'{MODEL_PATH}scaler.pkl'

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

# Verificar que pode ser recarregado
with open(scaler_path, 'rb') as f:
    scaler_check = pickle.load(f)

print(f'Scaler salvo em: {scaler_path}')
print(f'Verificação de recarga OK: mean={scaler_check.mean_.round(4)}')

Scaler salvo em: ../models/scaler.pkl
Verificação de recarga OK: mean=[94.5796  8.454 ]


---

## Resumo

### Artefatos gerados

| Artefato | Localização | Descrição |
|---|---|---|
| `X_train.csv` | `data/tabular/` | Features de treino (~70% dos registros) |
| `X_val.csv` | `data/tabular/` | Features de validação (~15%) |
| `X_test.csv` | `data/tabular/` | Features de teste (~15%) — **não usar antes da avaliação final** |
| `y_train.csv` | `data/tabular/` | Labels de treino |
| `y_val.csv` | `data/tabular/` | Labels de validação |
| `y_test.csv` | `data/tabular/` | Labels de teste |
| `scaler.pkl` | `models/` | StandardScaler treinado — necessário no serving |

### Features finais (9 colunas em X)

| Feature | Tipo | Transformação |
|---|---|---|
| `fever` | Binária | Nenhuma |
| `tachycardia` | Binária | Nenhuma |
| `crackles` | Binária | Nenhuma |
| `oxygen_saturation` | Contínua | StandardScaler (μ≈94.6, σ≈1.86) |
| `wbc_count` | Contínua | StandardScaler (μ≈8.46, σ≈1.27) |
| `chest_xray_result_effusion` | One-Hot | drop_first (ref: consolidation) |
| `chest_xray_result_infiltrate` | One-Hot | drop_first |
| `chest_xray_result_normal` | One-Hot | drop_first |
| `chest_xray_result_opacity` | One-Hot | drop_first |

### Decisões técnicas documentadas

- **Split por `patient_id`**: evita data leakage temporal (múltiplas notas por paciente)
- **`fit()` do scaler apenas no treino**: prevenção de leakage de distribuição
- **One-Hot com `drop_first=True`**: 5 categorias → 4 colunas (ref: `consolidation`)
- **Zero imputações**: dataset sem nulos (confirmado na EDA)
- **Zero remoções de outliers**: valores clinicamente válidos (confirmado na EDA)
- **Target como string**: scikit-learn suporta nativamente multi-classe sem `LabelEncoder`

---

## Próximo passo

**`04_escolha_modelo.ipynb`** — Etapa 2.4: Análise e justificativa dos algoritmos candidatos para o problema de classificação multi-classe (pneumonia / edema pulmonar / atelectasia).